# 🔬 TensorFlow Experiments — Nitrate Nowcasting
> Estación Yeguas · 5 arquitecturas · Walk-Forward Validation · Integrated Gradients

## 0 · Setup & Cargar datos

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.insert(0, 'src')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers, regularizers
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mlflow
import mlflow.tensorflow
import optuna
import joblib, json

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'TensorFlow {tf.__version__}')
print(f'MLflow     {mlflow.__version__}')
print(f'Optuna     {optuna.__version__}')
tf.random.set_seed(42)
np.random.seed(42)

# ── Paleta de colores por modelo ──────────────────────────────────────────────
COLORS = {'E_TCN': '#2196F3', 'F_Attention_BiLSTM': '#4CAF50', 'C_CNN_LSTM': '#FF9800', 'D_Transformer': '#9C27B0'}
MODEL_NAMES = {'E_TCN': 'TCN', 'F_Attention_BiLSTM': 'Attention', 'C_CNN_LSTM': '1D-CNN+LSTM', 'D_Transformer': 'Transformer'}


In [ ]:
# ── Configuración MLflow ──────────────────────────────────────────────────────
MLFLOW_EXPERIMENT = "nitrate_nowcasting"
MLFLOW_TRACKING_URI = "http://mlflow.example.com:5000"          # carpeta local; cambia a http://... para servidor remoto

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
print(f'Experimento : {MLFLOW_EXPERIMENT}')
print(f'Tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment ID: {exp.experiment_id}')
print()
print('Para visualizar los resultados, ejecuta en la terminal:')
print('  mlflow ui --port 5000')
print('  → http://mlflow.example.com:5000')

In [ ]:
# ── Carga y limpieza ──────────────────────────────────────────────────────────
df = pd.read_csv('data/processed/YEGUAS_data.csv')
df['_time'] = pd.to_datetime(df['_time'])
df = df.set_index('_time').sort_index()

# Eliminar columnas NITRATE excepto el target Y
cols_to_drop = [c for c in df.columns if 'NITRATE' in c and c != 'NITRATE_nitrate_no3_mg_l']
df = df.drop(columns=cols_to_drop)
df = df.dropna(subset=['NITRATE_nitrate_no3_mg_l'])
df = df.interpolate(method='linear').dropna()

# ── Eliminar variables colineales con conductivity ────────────────────────────
# TDS ≈ conductivity × factor; salinity ≈ f(conductivity) → información redundante
_colineal = [c for c in ['C4E_tds', 'C4E_salinity'] if c in df.columns]
if _colineal:
    df = df.drop(columns=_colineal)
    print(f'Columnas colineales eliminadas: {_colineal}')

# ── Ingeniería de variables: ciclo día/noche ──────────────────────────────────
_hour = df.index.hour
df['hour_sin'] = np.sin(2 * np.pi * _hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * _hour / 24)

print(f'Filas: {len(df)}  |  Features: {df.shape[1]-1}  |  Target: NITRATE_nitrate_no3_mg_l')
print(f'Rango: {df.index[0]}  →  {df.index[-1]}')
df.head(3)


## 1 · Preprocesado

In [ ]:
TARGET = 'NITRATE_nitrate_no3_mg_l'
FEATURE_COLS = [c for c in df.columns if c != TARGET]

# ── Hiperparámetros globales ──────────────────────────────────────────────────
WINDOW = 24         # pasos de 30 min = 12 h de historia
LAG_STEPS = [1, 2, 4]  # lags para Wide & Deep

# ── Separar X e Y ─────────────────────────────────────────────────────────────
y_raw = df[TARGET].values
X_raw = df[FEATURE_COLS].values

# ── Escalado ──────────────────────────────────────────────────────────────────
X_scaler = StandardScaler()
X_scaled = X_scaler.fit_transform(X_raw)

y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()

print(f'X escalado: {X_scaled.shape}  |  y escalado: {y_scaled.shape}')
print(f'Features  : {FEATURE_COLS}')


In [ ]:
# ── Función para crear ventanas temporales (modelos B/C/D) ────────────────────
def make_sequences(X, y, window):
    """Genera arrays 3D (n_samples, window, n_features) para modelos recurrentes."""
    Xs, ys = [], []
    for i in range(window - 1, len(X)):
        Xs.append(X[i - window + 1 : i + 1])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

# ── Función para crear lag features (modelo A Wide & Deep) ───────────────────
def make_lag_features(X, y, lags):
    """Añade columnas de lag. Retorna arrays 2D para MLP."""
    max_lag = max(lags)
    lag_arrays = [X[max_lag:]]
    for lag in lags:
        lag_arrays.append(X[max_lag - lag : len(X) - lag])
    X_with_lags = np.concatenate(lag_arrays, axis=1)
    y_aligned = y[max_lag:]
    return X_with_lags.astype(np.float32), y_aligned.astype(np.float32)

X_seq, y_seq = make_sequences(X_scaled, y_scaled, WINDOW)
X_lag, y_lag = make_lag_features(X_scaled, y_scaled, LAG_STEPS)

print(f'Secuencias  (B/C/D): {X_seq.shape}')
print(f'Lag features  (A):   {X_lag.shape}')

## 2 · Walk-Forward Validation — Definición de Folds

In [ ]:
def make_folds(n_samples, n_folds=5, min_train_ratio=0.6):
    """Genera índices para walk-forward expanding-window."""
    min_train = int(n_samples * min_train_ratio)
    remaining = n_samples - min_train
    fold_size = remaining // n_folds
    folds = []
    for k in range(n_folds):
        train_end = min_train + k * fold_size
        test_end  = train_end + fold_size
        if k == n_folds - 1:
            test_end = n_samples  # último fold usa todo lo que queda
        folds.append((slice(0, train_end), slice(train_end, test_end)))
    return folds

N_FOLDS = 5
folds_seq = make_folds(len(X_seq), N_FOLDS)
folds_lag = make_folds(len(X_lag), N_FOLDS)

print('Folds walk-forward (secuencias):')
for i, (tr, te) in enumerate(folds_seq):
    print(f'  Fold {i+1}: train[{tr.start}:{tr.stop}]  test[{te.start}:{te.stop}]')

## 5 · Modelo C — 1D-CNN + LSTM

In [ ]:
from model_c_cnn_lstm import build_cnn_lstm

build_cnn_lstm(WINDOW, X_seq.shape[2]).summary()


## 7 · HPO con Optuna + Walk-Forward Final

> **Objetivo**: encontrar el techo matemático del CNN+LSTM mediante Bayesian Optimization (TPE).  
> Cada trial de Optuna queda como un **nested run** dentro del run padre `HPO_CNN_LSTM` en MLflow.

| Hiperparámetro | Espacio de búsqueda |
|---|---|
| `conv_filters` | {16, 32, 64} |
| `kernel_size` | {3, 6, 12} |
| `lstm_units` | {32, 64, 128} |
| `dropout` | [0.1, 0.4] continuo |
| `lr` (Adam) | [1e-4, 1e-2] log-uniforme |


In [ ]:
def run_walk_forward(model_fn, X, y, folds, build_kwargs, epochs=100, batch_size=32):
    """
    Ejecuta walk-forward validation para un model_fn dado.
    Si hay un mlflow.active_run(), registra métricas por fold (step) y resumen.
    Retorna DataFrame con métricas por fold y todas las predicciones.
    """
    results = []
    all_preds, all_true = [], []

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=0)

    for fold_idx, (tr_sl, te_sl) in enumerate(folds):
        X_tr, y_tr = X[tr_sl], y[tr_sl]
        X_te, y_te = X[te_sl], y[te_sl]

        model = model_fn(**build_kwargs)
        model.fit(
            X_tr, y_tr,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=0.15,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        y_pred_scaled = model.predict(X_te, verbose=0).flatten()
        y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        y_true = y_scaler.inverse_transform(y_te.reshape(-1, 1)).flatten()

        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2   = r2_score(y_true, y_pred)

        # ── MLflow: métricas por fold como pasos de la curva ──────────────────
        if mlflow.active_run():
            mlflow.log_metrics(
                {"fold_mae": float(mae), "fold_rmse": float(rmse), "fold_r2": float(r2)},
                step=fold_idx + 1
            )

        results.append({'Fold': fold_idx + 1, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
        all_preds.extend(y_pred)
        all_true.extend(y_true)
        print(f'  Fold {fold_idx+1}/{len(folds)}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}')

    df_results = pd.DataFrame(results)

    # ── MLflow: métricas resumen del experimento ───────────────────────────────
    if mlflow.active_run():
        last3 = df_results.tail(3)
        mlflow.log_metrics({
            "mean_mae":   float(df_results['MAE'].mean()),
            "mean_rmse":  float(df_results['RMSE'].mean()),
            "mean_r2":    float(df_results['R2'].mean()),
            "last3_mae":  float(last3['MAE'].mean()),
            "last3_rmse": float(last3['RMSE'].mean()),
            "last3_r2":   float(last3['R2'].mean()),
        })

    return df_results, np.array(all_true), np.array(all_preds)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7A · Importar arquitectura CNN+LSTM parametrizable
# ══════════════════════════════════════════════════════════════════════════════
from model_c_cnn_lstm import build_cnn_lstm_hpo

# ══════════════════════════════════════════════════════════════════════════════
# 7B · Función objetivo Optuna + estudio con MLflow nested runs
# ══════════════════════════════════════════════════════════════════════════════

# Usar 3 folds durante HPO para velocidad; la evaluación final usará los 5
HPO_FOLDS = 3

def objective(trial):
    hp = {
        'conv_filters': trial.suggest_categorical('conv_filters', [16, 32, 64]),
        'kernel_size':  trial.suggest_categorical('kernel_size',  [3, 6, 12]),
        'lstm_units':   trial.suggest_categorical('lstm_units',   [32, 64, 128]),
        'dropout':      trial.suggest_float('dropout', 0.1, 0.4),
        'lr':           trial.suggest_float('lr', 1e-4, 1e-2, log=True),
    }

    def build_fn(window, n_features):
        return build_cnn_lstm_hpo(window, n_features, **hp)

    with mlflow.start_run(run_name=f'trial_{trial.number:03d}', nested=True):
        mlflow.log_params({**hp, 'trial': trial.number, 'hpo_folds': HPO_FOLDS})

        r, _, _ = run_walk_forward(
            model_fn=build_fn,
            X=X_seq, y=y_seq,
            folds=folds_seq[:HPO_FOLDS],
            build_kwargs={'window': WINDOW, 'n_features': X_seq.shape[2]},
            epochs=80,
            batch_size=32,
        )
        mean_mae = float(r['MAE'].mean())
        mlflow.log_metric('hpo_mean_mae', mean_mae)

    return mean_mae


# ══════════════════════════════════════════════════════════════════════════════
# 7C · Lanzar el estudio — run padre en MLflow contiene todos los trials
# ══════════════════════════════════════════════════════════════════════════════

N_TRIALS = 30

with mlflow.start_run(
    run_name='HPO_CNN_LSTM',
    tags={'phase': 'hpo', 'architecture': 'CNN+RNN'},
) as hpo_run:
    mlflow.log_params({
        'n_trials':     N_TRIALS,
        'hpo_folds':    HPO_FOLDS,
        'window':       WINDOW,
        'n_features':   X_seq.shape[2],
        'search_space': 'conv_filters[16,32,64] | kernel[3,6,12] | lstm[32,64,128] | dropout[0.1-0.4] | lr[1e-4-1e-2]',
        'sampler':      'TPE',
    })

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
        study_name='cnn_lstm_hpo',
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_params = study.best_params
    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
    mlflow.log_metric('best_hpo_mae', study.best_value)
    hpo_run_id = hpo_run.info.run_id

print(f'\n── Mejor trial: #{study.best_trial.number} / {N_TRIALS} ──')
print(f'   MAE medio HPO ({HPO_FOLDS} folds): {study.best_value:.4f} mg/L')
print(f'\n── Parámetros óptimos ──')
for k, v in best_params.items():
    print(f'   {k:<20s}: {v}')
print(f'\nMLflow HPO parent run: {hpo_run_id}')
print(f'→ Cada trial es un nested run visible en MLflow UI bajo HPO_CNN_LSTM')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7D · Evaluación final — mejores hiperparámetros con los 5 folds completos
# ══════════════════════════════════════════════════════════════════════════════
print('='*60)
print('CNN+LSTM — Evaluación final con parámetros HPO óptimos')
print('='*60)
print(f'Parámetros: {best_params}\n')

def build_best(window, n_features):
    return build_cnn_lstm_hpo(window, n_features, **best_params)

with mlflow.start_run(
    run_name='C_CNN_LSTM_best',
    tags={'phase': 'final_eval', 'architecture': 'CNN+RNN'},
) as run_C:
    mlflow.log_params({
        **best_params,
        'model_type':  '1D_CNN_LSTM_HPO',
        'window':      WINDOW,
        'n_features':  X_seq.shape[2],
        'n_folds':     N_FOLDS,
        'hpo_run_id':  hpo_run_id,
        'epochs':      100,
        'batch_size':  32,
    })

    r_C, true_C, pred_C = run_walk_forward(
        model_fn=build_best,
        X=X_seq, y=y_seq,
        folds=folds_seq,
        build_kwargs={'window': WINDOW, 'n_features': X_seq.shape[2]},
        epochs=100,
        batch_size=32,
    )

    # Reentrenar sobre todos los datos para el modelo de producción
    _es = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)
    final_C = build_best(WINDOW, X_seq.shape[2])
    final_C.fit(X_seq, y_seq, epochs=150, batch_size=32,
                validation_split=0.1, callbacks=[_es], verbose=0)
    mlflow.tensorflow.log_model(final_C, 'model')
    run_id_C   = run_C.info.run_id
    run_name_C = 'C_CNN_LSTM_best'

print(r_C.to_string(index=False))
mae_final = r_C['MAE'].tail(3).mean()
print(f'\nBaseline v1 (fijo):    MAE = 4.26 mg/L')
print(f'Post-HPO (últimos 3):  MAE = {mae_final:.3f} mg/L  '
      f'({"↓ mejora" if mae_final < 4.26 else "↑ sin mejora"} vs baseline)')
print(f'\nMLflow Run ID: {run_id_C}')



## 14 · Sección E · CNN+BiLSTM v2 — Reducir MAE por debajo de los folds críticos

> **Diagnóstico**: MAE de folds 4–5 (~7–11 mg/L) revela *distribution shift*. La HPO original solo usó 3 folds y optimizó para datos estables.

| Estrategia | Descripción |
|---|---|
| **E1** Ventana 48 pasos | 24 h de historia → captura ciclos diarios completos |
| **E2** Lags NO3 (1,2,4,8) | Autoregresión implícita del target |
| **E3** Rolling mean/std | Tendencias locales de conductividad y NO3 |
| **E4** Arquitectura v2 | CNN dilated + BiLSTM + Atención + Huber loss |
| **E5** HPO con 5 folds | Optimización sobre los períodos difíciles también |


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 14A · Ingeniería de features v2 + secuencias con ventana de 48 pasos (24 h)
# ══════════════════════════════════════════════════════════════════════════════
# Diagnóstico del problema:
#   Fold 4 MAE≈6.9 / Fold 5 MAE≈10.9 → distribution shift en períodos tardíos
#   Causas: ventana corta (12 h), sin features autoregresivas, HPO solo en 3 folds
#
# Estrategias:
#   E1. Ventana 48 pasos (24 h) → captura ciclos diarios completos
#   E2. Lags del target (1,2,4,8) → autoregresión implícita
#   E3. Rolling mean/std de conductividad y NO3 → tendencias locales
#   E4. Arquitectura v2: CNN dilated + BiLSTM + Atención + Huber loss
#   E5. HPO sobre los 5 folds (no solo 3) → optimiza también los períodos difíciles

def enrich_features(df_in, target_col='NITRATE_nitrate_no3_mg_l',
                    lags=(1, 2, 4, 8), roll_windows=(4, 8)):
    """Añade lags del target, rolling stats y derivada discreta al DataFrame."""
    df = df_in.copy()

    # Lags del target (autoregresión implícita)
    for k in lags:
        df[f'no3_lag_{k}'] = df[target_col].shift(k)

    # Rolling stats de conductividad y NO3
    for w in roll_windows:
        df[f'cond_rmean_{w}'] = df['C4E_conductivity'].rolling(w, min_periods=1).mean()
        df[f'cond_rstd_{w}']  = df['C4E_conductivity'].rolling(w, min_periods=1).std().fillna(0)
        df[f'no3_rmean_{w}']  = df[target_col].rolling(w, min_periods=1).mean()

    # Derivada discreta
    df['cond_delta'] = df['C4E_conductivity'].diff().fillna(0)
    df['no3_delta']  = df[target_col].diff().fillna(0)

    return df.ffill().fillna(0)


WINDOW_V2 = 48          # 24 h de historia (48 × 30 min)

df_v2         = enrich_features(df)
FEATURE_V2    = [c for c in df_v2.columns if c != TARGET]

y_raw_v2      = df_v2[TARGET].values
X_raw_v2      = df_v2[FEATURE_V2].values

X_scaler_v2   = StandardScaler()
X_scaled_v2   = X_scaler_v2.fit_transform(X_raw_v2)

# y_scaler_v2 es equivalente a y_scaler (mismo target), se define para claridad
y_scaler_v2   = MinMaxScaler()
y_scaled_v2   = y_scaler_v2.fit_transform(y_raw_v2.reshape(-1, 1)).flatten()

X_seq_v2, y_seq_v2 = make_sequences(X_scaled_v2, y_scaled_v2, WINDOW_V2)
folds_v2            = make_folds(len(X_seq_v2), N_FOLDS)

print(f'Features  v1 : {len(FEATURE_COLS)}  →  v2: {len(FEATURE_V2)}')
print(f'Nuevas        : {[f for f in FEATURE_V2 if f not in FEATURE_COLS]}')
print(f'Secuencias v2 : {X_seq_v2.shape}  (WINDOW={WINDOW_V2})')
print(f'\nFolds walk-forward (v2):')
for i, (tr, te) in enumerate(folds_v2):
    print(f'  Fold {i+1}: train[{tr.start}:{tr.stop}]  test[{te.start}:{te.stop}]')


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 14B · Arquitectura v2: CNN dilated + BiLSTM + Atención + Huber loss
# ══════════════════════════════════════════════════════════════════════════════

def build_cnn_lstm_v2(window, n_features,
                      conv_filters=32, kernel_size=6,
                      lstm_units=64, dropout=0.2, lr=1e-3):
    """
    Mejoras sobre C_CNN_LSTM:
      1. Dos ramas CNN (dilation=1 y dilation=2) → patrones a distintas escalas
      2. BiLSTM → usa contexto pasado y futuro dentro de la ventana
      3. MultiHeadAttention (4 cabezas) → pondera pasos temporales relevantes
      4. Huber loss → robusta a outliers de los folds tardíos
      5. clipnorm en Adam → estabiliza el entrenamiento con gradientes grandes
    """
    inp = keras.Input(shape=(window, n_features), name='input')

    # Rama 1: CNN causal estándar
    x1 = layers.Conv1D(conv_filters, kernel_size=kernel_size,
                       activation='relu', padding='causal', name='conv_s1')(inp)
    x1 = layers.Conv1D(conv_filters * 2, kernel_size=kernel_size,
                       activation='relu', padding='causal', name='conv_s2')(x1)
    x1 = layers.MaxPooling1D(pool_size=2, name='pool1')(x1)
    x1 = layers.Dropout(dropout * 0.5)(x1)

    # Rama 2: CNN causal dilatada (captura dependencias más largas)
    x2 = layers.Conv1D(conv_filters, kernel_size=kernel_size,
                       activation='relu', padding='causal',
                       dilation_rate=2, name='conv_d1')(inp)
    x2 = layers.Conv1D(conv_filters * 2, kernel_size=kernel_size,
                       activation='relu', padding='causal',
                       dilation_rate=2, name='conv_d2')(x2)
    x2 = layers.MaxPooling1D(pool_size=2, name='pool2')(x2)
    x2 = layers.Dropout(dropout * 0.5)(x2)

    # Fusionar ramas
    x = layers.Add(name='merge_cnn')([x1, x2])

    # BiLSTM: contexto pasado y futuro dentro de la ventana
    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True,
                    kernel_regularizer=regularizers.l2(1e-4)),
        name='bilstm'
    )(x)
    x = layers.Dropout(dropout)(x)

    # Self-Attention
    key_d = max(16, lstm_units // 4)
    attn  = layers.MultiHeadAttention(
        num_heads=4, key_dim=key_d, dropout=dropout * 0.5, name='self_attn'
    )(x, x)
    x = layers.Add(name='res_attn')([x, attn])
    x = layers.LayerNormalization(name='ln')(x)

    # Cabeza de regresión
    x   = layers.GlobalAveragePooling1D(name='gap')(x)
    x   = layers.Dense(64, activation='relu',
                       kernel_regularizer=regularizers.l2(1e-4), name='fc1')(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(32, activation='relu', name='fc2')(x)
    out = layers.Dense(1, name='output')(x)

    model = keras.Model(inp, out, name='E_CNN_BiLSTM_v2')
    model.compile(
        optimizer=keras.optimizers.Adam(lr, clipnorm=1.0),
        loss=keras.losses.Huber(delta=1.0),
        metrics=['mae'],
    )
    return model

build_cnn_lstm_v2(WINDOW_V2, X_seq_v2.shape[2]).summary()


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 14C · HPO v2 — Optuna con los 5 folds (optimiza también los períodos difíciles)
# ══════════════════════════════════════════════════════════════════════════════
N_TRIALS_V2 = 40

def objective_v2(trial):
    hp = {
        'conv_filters': trial.suggest_categorical('conv_filters', [16, 32, 64]),
        'kernel_size':  trial.suggest_categorical('kernel_size',  [3, 6, 12]),
        'lstm_units':   trial.suggest_categorical('lstm_units',   [32, 64, 128]),
        'dropout':      trial.suggest_float('dropout', 0.1, 0.4),
        'lr':           trial.suggest_float('lr', 5e-5, 5e-3, log=True),
    }

    def build_fn(window, n_features):
        return build_cnn_lstm_v2(window, n_features, **hp)

    with mlflow.start_run(run_name=f'E_trial_{trial.number:03d}', nested=True):
        mlflow.log_params({**hp, 'trial': trial.number, 'hpo_folds': 'all5'})

        r, _, _ = run_walk_forward(
            model_fn=build_fn,
            X=X_seq_v2, y=y_seq_v2,
            folds=folds_v2,
            build_kwargs={'window': WINDOW_V2, 'n_features': X_seq_v2.shape[2]},
            epochs=80,
            batch_size=32,
        )
        # Métrica: MAE de los últimos 3 folds (los más difíciles)
        last3_mae = float(r['MAE'].tail(3).mean())
        mlflow.log_metric('hpo_last3_mae', last3_mae)

    return last3_mae


with mlflow.start_run(
    run_name='HPO_E_CNN_BiLSTM',
    tags={'phase': 'hpo_v2', 'architecture': 'CNN+BiLSTM'},
) as hpo_run_v2:
    mlflow.log_params({
        'n_trials':     N_TRIALS_V2,
        'window':       WINDOW_V2,
        'n_features':   X_seq_v2.shape[2],
        'search_space': 'conv_filters[16,32,64] | kernel[3,6,12] | lstm[32,64,128] | dropout[0.1-0.4] | lr[5e-5-5e-3]',
        'sampler':      'TPE',
        'hpo_folds':    'all5',
    })

    study_v2 = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
        study_name='cnn_bilstm_v2_hpo',
    )
    study_v2.optimize(objective_v2, n_trials=N_TRIALS_V2, show_progress_bar=True)

    best_params_v2 = study_v2.best_params
    mlflow.log_params({f'best_{k}': v for k, v in best_params_v2.items()})
    mlflow.log_metric('best_last3_mae', study_v2.best_value)
    hpo_run_id_v2 = hpo_run_v2.info.run_id

print(f'\n── Mejor trial: #{study_v2.best_trial.number} / {N_TRIALS_V2} ──')
print(f'   MAE últimos 3 folds: {study_v2.best_value:.4f} mg/L')
print(f'\n── Parámetros óptimos ──')
for k, v in best_params_v2.items():
    print(f'   {k:<20s}: {v}')
print(f'\nMLflow HPO parent run: {hpo_run_id_v2}')


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 14D · Evaluación final — 5 folds + modelo de producción
# ══════════════════════════════════════════════════════════════════════════════
print('='*60)
print('E_CNN_BiLSTM_v2 — Evaluación final con hiperparámetros óptimos')
print('='*60)
print(f'Parámetros: {best_params_v2}\n')

def build_best_v2(window, n_features):
    return build_cnn_lstm_v2(window, n_features, **best_params_v2)

with mlflow.start_run(
    run_name='E_CNN_BiLSTM_v2_best',
    tags={'phase': 'final_eval', 'architecture': 'CNN+BiLSTM_v2'},
) as run_Ev2:
    mlflow.log_params({
        **best_params_v2,
        'model_type':  'CNN_BiLSTM_v2',
        'window':      WINDOW_V2,
        'n_features':  X_seq_v2.shape[2],
        'n_folds':     N_FOLDS,
        'hpo_run_id':  hpo_run_id_v2,
        'epochs':      120,
        'batch_size':  32,
    })

    r_Ev2, true_Ev2, pred_Ev2 = run_walk_forward(
        model_fn=build_best_v2,
        X=X_seq_v2, y=y_seq_v2,
        folds=folds_v2,
        build_kwargs={'window': WINDOW_V2, 'n_features': X_seq_v2.shape[2]},
        epochs=120,
        batch_size=32,
    )

    # Modelo de producción reentrenado sobre todos los datos
    _es = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=25, restore_best_weights=True, verbose=0)
    final_Ev2 = build_best_v2(WINDOW_V2, X_seq_v2.shape[2])
    final_Ev2.fit(
        X_seq_v2, y_seq_v2,
        epochs=200, batch_size=32,
        validation_split=0.1,
        callbacks=[_es],
        verbose=1,
    )
    mlflow.tensorflow.log_model(final_Ev2, 'model')
    run_id_Ev2   = run_Ev2.info.run_id
    run_name_Ev2 = 'E_CNN_BiLSTM_v2_best'

print(r_Ev2.to_string(index=False))
mae_v2 = r_Ev2['MAE'].tail(3).mean()
try:
    mae_baseline = r_C['MAE'].tail(3).mean()
    delta = mae_baseline - mae_v2
    print(f'\nBaseline C_CNN_LSTM  (últimos 3): MAE = {mae_baseline:.3f} mg/L')
    print(f'E_CNN_BiLSTM_v2      (últimos 3): MAE = {mae_v2:.3f} mg/L  '
          f'({"↓ MEJORA" if delta > 0 else "↑ sin mejora"}: {delta / mae_baseline * 100:+.1f}%)')
except NameError:
    print(f'\nE_CNN_BiLSTM_v2 (últimos 3): MAE = {mae_v2:.3f} mg/L')
print(f'MLflow Run ID: {run_id_Ev2}')


## 8 · Resultados HPO y evaluación final


In [ ]:
# ── Recuperar solo los runs de evaluación final (excluye trials HPO) ──────────
runs_df = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string="tags.phase = 'final_eval'",
    order_by=["metrics.last3_mae ASC"],
)

display_cols = [
    'tags.mlflow.runName',
    'metrics.mean_mae',  'metrics.mean_rmse',  'metrics.mean_r2',
    'metrics.last3_mae', 'metrics.last3_rmse', 'metrics.last3_r2',
]
avail = [c for c in display_cols if c in runs_df.columns]
print('\n── Tabla — Evaluación Final post-HPO (desde MLflow) ──')
print(runs_df[avail].to_string(index=False))

best_run_row  = runs_df.iloc[0]
best_run_id   = best_run_row['run_id']
best_run_name = best_run_row['tags.mlflow.runName']

print(f'\n🏆 Modelo ganador : {best_run_name}')
print(f'   MAE últimos 3  : {best_run_row["metrics.last3_mae"]:.3f} mg/L')
print(f'   R² últimos 3   : {best_run_row["metrics.last3_r2"]:.4f}')
print(f'   Run ID         : {best_run_id}')
print(f'\n   mlflow ui → http://127.0.0.1:5000  (pestaña Experiments → {MLFLOW_EXPERIMENT})')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Izquierda: convergencia HPO ────────────────────────────────────────────────
try:
    trial_nums  = [t.number for t in study.trials if t.value is not None]
    trial_maes  = [t.value  for t in study.trials if t.value is not None]
    best_so_far = np.minimum.accumulate(trial_maes)

    axes[0].scatter(trial_nums, trial_maes, alpha=0.45, s=25,
                    color='#9C27B0', zorder=2, label='Trial MAE')
    axes[0].plot(trial_nums, best_so_far, color='#FF5722', linewidth=2,
                 label='Mejor acumulado', zorder=3)
    axes[0].axhline(4.26, color='#9E9E9E', linestyle='--',
                    linewidth=1.2, label='Baseline v1 (4.26 mg/L)')
    axes[0].scatter([study.best_trial.number], [study.best_value],
                    color='gold', s=150, zorder=4, marker='*',
                    label=f'Óptimo (trial #{study.best_trial.number})')
    axes[0].set_xlabel('Trial #')
    axes[0].set_ylabel('MAE medio mg/L')
    axes[0].set_title(f'Convergencia HPO — Optuna TPE\n({N_TRIALS} trials · {HPO_FOLDS} folds · EarlyStopping)',
                      fontsize=11)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)
except NameError:
    axes[0].text(0.5, 0.5, 'HPO no ejecutado\n(ejecuta la celda 17 primero)',
                 ha='center', va='center', fontsize=12, color='gray',
                 transform=axes[0].transAxes)
    axes[0].set_title('Convergencia HPO — sin datos', fontsize=11)

# ── Derecha: predicciones CNN+LSTM ────────────────────────────────────────────
try:
    axes[1].plot(true_C[-200:], label='Real',     color='#333', linewidth=1.5, alpha=0.8)
    axes[1].plot(pred_C[-200:], label='Predicho', color='#FF9800',
                 linewidth=1.5, alpha=0.9, linestyle='--')
    mae_last3 = r_C['MAE'].tail(3).mean()
    r2_last3  = r_C['R2'].tail(3).mean()
    axes[1].set_title(f'CNN+LSTM — Mejores hiperparámetros\nMAE={mae_last3:.3f} mg/L  R²={r2_last3:.4f}  [últimos 3 folds]',
                      fontsize=11)
    axes[1].set_xlabel('Muestra (últimas 200)')
    axes[1].set_ylabel('NO₃⁻ (mg/L)')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)
except NameError:
    axes[1].text(0.5, 0.5, 'CNN+LSTM no ejecutado\n(ejecuta la celda 18 primero)',
                 ha='center', va='center', fontsize=12, color='gray',
                 transform=axes[1].transAxes)
    axes[1].set_title('CNN+LSTM Predicciones — sin datos', fontsize=11)

plt.suptitle('HPO Results — CNN+LSTM Nitrate Nowcasting\nEstación Yeguas', fontsize=13)
plt.tight_layout()
_plot_path = 'outputs/hpo_results.png'
plt.savefig(_plot_path, dpi=150, bbox_inches='tight')
plt.show()

try:
    with mlflow.start_run(run_id=run_id_C):
        mlflow.log_artifact(_plot_path, artifact_path='plots')
    print(f'Gráfico guardado y registrado en MLflow (run: {run_name_C})')
except NameError:
    print(f'Gráfico guardado en {_plot_path} (sin registrar en MLflow — run_id_C no disponible)')


In [ ]:
# Evolución de MAE y R² por fold — modelo campeón post-HPO
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(r_C['Fold'], r_C['MAE'], marker='o', color='#FF9800',
             linewidth=2, label='CNN+LSTM (HPO)')
axes[1].plot(r_C['Fold'], r_C['R2'],  marker='o', color='#FF9800',
             linewidth=2, label='CNN+LSTM (HPO)')

for ax, title, ylabel in [
    (axes[0], 'MAE por Fold (Walk-Forward)', 'MAE (mg/L)'),
    (axes[1], 'R² por Fold (Walk-Forward)',  'R²'),
]:
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Fold')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(1, N_FOLDS + 1))

plt.suptitle('Walk-Forward Validation — CNN+LSTM post-HPO', fontsize=13)
plt.tight_layout()
_folds_plot_path = 'outputs/walk_forward_folds.png'
plt.savefig(_folds_plot_path, dpi=150, bbox_inches='tight')
plt.show()

with mlflow.start_run(run_id=run_id_C):
    mlflow.log_artifact(_folds_plot_path, artifact_path='plots')
print('Gráfico de folds guardado y registrado en MLflow.')


## 9 · Integrated Gradients — Interpretabilidad del mejor modelo

> Implementado con `tf.GradientTape` nativo, sin dependencias externas.

In [ ]:
def integrated_gradients(model, inputs, baseline=None, steps=50):
    """
    Calcula Integrated Gradients para un lote de inputs.
    
    Args:
        model   : modelo Keras compilado
        inputs  : np.array (batch, ...) — observaciones a explicar
        baseline: np.array misma forma que inputs — punto de referencia (0 = promedio nulo)
        steps   : número de pasos de integración (mayor → más preciso)
    Returns:
        ig : np.array misma forma que inputs — atribución de cada feature
    """
    inputs   = tf.cast(inputs,   dtype=tf.float32)
    if baseline is None:
        baseline = tf.zeros_like(inputs)
    baseline = tf.cast(baseline, dtype=tf.float32)

    # Interpolación lineal entre baseline e input
    alphas = tf.linspace(0.0, 1.0, steps + 1)[:, tf.newaxis]  # (steps+1, 1)

    # Para cada input del batch
    all_ig = []
    for idx in range(len(inputs)):
        x       = inputs[idx:idx+1]   # (1, ...)
        base    = baseline[idx:idx+1]

        # Genera path de puntos interpolados
        if x.ndim == 2:
            interp = base + alphas[:, tf.newaxis] * (x - base)   # (steps+1, 1, feat)
            interp = tf.reshape(interp, (-1, x.shape[-1]))
        else:  # 3D (window, feat)
            interp = base + alphas[:, tf.newaxis, tf.newaxis] * (x - base)  # (steps+1, 1, w, f)
            interp = tf.reshape(interp, (-1, x.shape[-2], x.shape[-1]))

        with tf.GradientTape() as tape:
            tape.watch(interp)
            preds = model(interp, training=False)

        grads    = tape.gradient(preds, interp)          # (steps+1, ...)
        avg_grad = tf.reduce_mean(grads, axis=0)         # (...)
        ig       = (x[0] - base[0]) * avg_grad          # (...)
        all_ig.append(ig.numpy())

    return np.array(all_ig)

In [ ]:
# ── Cargar el modelo ganador directamente desde MLflow ────────────────────────
print(f'Cargando modelo ganador: {best_run_name}  (run_id: {best_run_id})')

best_model_uri = f"runs:/{best_run_id}/model"
best_model     = mlflow.tensorflow.load_model(best_model_uri)

# Determinar qué array de entrada usa (secuencial vs lag features)
is_seq_model = (best_run_name != "A_WideDeep")
X_all = X_seq if is_seq_model else X_lag
y_all = y_seq if is_seq_model else y_lag

print(f'Modelo cargado: {best_model.name}')
print(f'Input shape  : {best_model.input_shape}')
print(f'Datos para IG: {X_all.shape}')

In [ ]:
# ── Calcular IG sobre 50 muestras aleatorias ──────────────────────────────────
np.random.seed(42)
sample_idx = np.random.choice(len(X_all), size=50, replace=False)
X_sample   = X_all[sample_idx].astype(np.float32)

ig_values  = integrated_gradients(best_model, X_sample, steps=50)

# Colapsar dimensión temporal si el modelo es secuencial
if ig_values.ndim == 3:
    ig_flat = np.abs(ig_values).mean(axis=1)  # media sobre pasos de tiempo → (50, n_features)
else:
    ig_flat = np.abs(ig_values)[:, :len(FEATURE_COLS)]

mean_ig = ig_flat.mean(axis=0)[:len(FEATURE_COLS)]  # media sobre muestras

# ── Gráfico de importancia ─────────────────────────────────────────────────────
sorted_idx   = np.argsort(mean_ig)[::-1]
top_features = [FEATURE_COLS[i] for i in sorted_idx]
top_values   = mean_ig[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_features[::-1], top_values[::-1],
        color=COLORS.get(best_run_name, '#607D8B'), edgecolor='white', alpha=0.85)
ax.set_xlabel('Importancia media (|Integrated Gradients|)', fontsize=11)
ax.set_title(f'Integrated Gradients — {best_run_name}\n'
             'Contribución de cada feature a la predicción de NO₃⁻', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
_ig_plot_path = 'outputs/integrated_gradients.png'
plt.savefig(_ig_plot_path, dpi=150, bbox_inches='tight')
plt.show()

# ── Registrar importancias y gráfico en MLflow ────────────────────────────────
ig_dict = {f"ig_{feat}": float(val) for feat, val in zip(FEATURE_COLS, mean_ig)}
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metrics(ig_dict)
    mlflow.log_artifact(_ig_plot_path, artifact_path="interpretability")

print('── Top 5 features más influyentes ──')
for i in range(5):
    print(f'  {i+1}. {top_features[i]:<35s}  IG={top_values[i]:.5f}')
print(f'\nIntegrated Gradients registrados en MLflow run: {best_run_id}')


## 10 · Registro en MLflow Model Registry y exportación de artefactos

In [ ]:
# ── Registrar el modelo ganador en el MLflow Model Registry ───────────────────
REGISTRY_NAME = "nitrate_nowcasting_best_01"

model_uri       = f"runs:/{best_run_id}/model"
registered_mv   = mlflow.register_model(model_uri, REGISTRY_NAME)

print(f'Modelo registrado en Model Registry:')
print(f'  Nombre  : {registered_mv.name}')
print(f'  Versión : {registered_mv.version}')
print(f'  Run ID  : {best_run_id}')

# ── Guardar escaladores y metadatos como artefactos del run ganador ────────────
os.makedirs('outputs/artifacts', exist_ok=True)

joblib.dump(X_scaler, 'outputs/artifacts/X_scaler.pkl')
joblib.dump(y_scaler, 'outputs/artifacts/y_scaler.pkl')

best_metrics = runs_df.iloc[0]
meta = {
    'model_name':      REGISTRY_NAME,
    'model_version':   registered_mv.version,
    'run_name':        best_run_name,
    'run_id':          best_run_id,
    'window':          WINDOW,
    'lag_steps':       LAG_STEPS,
    'features':        FEATURE_COLS,
    'mae_last3_folds': float(best_metrics['metrics.last3_mae']),
    'r2_last3_folds':  float(best_metrics['metrics.last3_r2']),
    'created':         pd.Timestamp.now().isoformat(),
}
with open('outputs/artifacts/meta.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact('outputs/artifacts/X_scaler.pkl', artifact_path='scalers')
    mlflow.log_artifact('outputs/artifacts/y_scaler.pkl', artifact_path='scalers')
    mlflow.log_artifact('outputs/artifacts/meta.json',    artifact_path='metadata')

print('\n── Artefactos registrados en MLflow ──')
print('  scalers/X_scaler.pkl')
print('  scalers/y_scaler.pkl')
print('  metadata/meta.json')
print()
print('── Metadatos ──')
print(json.dumps(meta, indent=2, ensure_ascii=False))
print()
print('Para cargar el modelo en producción:')
print(f'  model = mlflow.tensorflow.load_model("models:/{REGISTRY_NAME}/latest")')
print('  # o por versión específica:')
print(f'  model = mlflow.tensorflow.load_model("models:/{REGISTRY_NAME}/{registered_mv.version}")')


## 11 · Guía de uso MLflow

### Visualizar experimentos
```bash
# En la terminal, desde la raíz del proyecto:
mlflow ui --port 5000
# → http://127.0.0.1:5000
```

### Comparar runs programáticamente
```python
import mlflow

runs = mlflow.search_runs(
    experiment_names=["nitrate_nowcasting"],
    order_by=["metrics.last3_mae ASC"]
)
print(runs[['tags.mlflow.runName','metrics.last3_mae','metrics.last3_r2']].to_string())
```

### Cargar modelo ganador para inferencia
```python
import mlflow.tensorflow, joblib, numpy as np

# Desde Model Registry (siempre apunta a la mejor versión)
model     = mlflow.tensorflow.load_model("models:/nitrate_nowcasting_best_01/latest")
X_scaler  = joblib.load("outputs/artifacts/X_scaler.pkl")
y_scaler  = joblib.load("outputs/artifacts/y_scaler.pkl")

# Inferencia (requiere ventana de 96 pasos × n_features)
X_new     = X_scaler.transform(raw_sensor_data)          # (96, n_features)
X_input   = X_new[np.newaxis, ...]                        # (1, 96, n_features)
pred_norm = model.predict(X_input).flatten()
pred_mgL  = y_scaler.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
print(f"NO₃⁻ estimado: {pred_mgL[0]:.2f} mg/L")
```

### Estructura del proyecto
```
NITRATE_model/
├── data/
│   ├── raw/          ← CSV originales del sensor
│   └── processed/    ← YEGUAS_data.csv (limpio)
├── outputs/
│   ├── artifacts/    ← escaladores (.pkl) y meta.json
│   ├── model_comparison.png
│   ├── walk_forward_folds.png
│   └── integrated_gradients.png
├── src/
│   ├── csv_cleaner.py
│   ├── main.py
│   └── model_trainning.py
├── tf_experiments.ipynb
├── model_selection.md
├── pyproject.toml
└── README.md
```

### Flujo completo MLflow
```
tf_experiments.ipynb
  ├── Setup + MLflow Experiment config
  ├── Run A/B/C/D  → mlflow.start_run() por cada arquitectura
  │     ├── mlflow.log_params()          # hiperparámetros
  │     ├── mlflow.log_metrics(step=k)   # MAE/RMSE/R² por fold
  │     ├── mlflow.log_metrics()         # resumen mean/last3
  │     └── mlflow.tensorflow.log_model()# modelo final .tf
  ├── Comparativa  → mlflow.search_runs() ordena por last3_mae
  ├── Gráficos     → mlflow.log_artifact() en run ganador
  ├── Integrated Gradients → mlflow.log_metrics(ig_<feature>)
  └── Model Registry → mlflow.register_model() "nitrate_nowcasting_best_01"
```
